# 🔁 LLM Lie Detector: research loop

Claude improves a FIBS lie-detection probe in a loop: **propose → train → evaluate on unseen lie families → keep or revert → repeat**.

1. `Runtime → Change runtime type → GPU` (T4 is fine, L4/A100 faster)
2. Add your Anthropic API key under **🔑 Secrets** (left sidebar) as `ANTHROPIC_API_KEY`, notebook access ON
3. Run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = 'people-looking-at-my-laptop-liked-it'
!rm -rf LLM_lie_detector && git clone -q -b $BRANCH https://github.com/msverma101/LLM_lie_detector.git
%cd LLM_lie_detector/experiments/research_loop
!git log --oneline -1

In [ ]:
# Colab already has torch, pandas, numpy; don't upgrade pandas (google-colab pins it)
!pip install -q -U transformers accelerate anthropic huggingface_hub

## Step 1: cache activations (the only GPU step, ~20 min on a T4)
Resumable: if Colab disconnects, rerun this cell. Run the quick test first.

In [ ]:
# QUICK TEST first (~220 rows, a few minutes): checks the whole pipeline end to end
!python cache_activations.py --model Qwen/Qwen3.5-2B --n_per_family 8 --out acts_test && python eval.py --acts acts_test

In [ ]:
MODEL = 'Qwen/Qwen3.5-2B'   # fallback if too slow: 'Qwen/Qwen3.5-0.8B'
N_PER_FAMILY = 200          # ~5,600 rows, ~20 min on a T4
!python cache_activations.py --model $MODEL --n_per_family $N_PER_FAMILY --out acts

## Step 2: baseline (the FIBS paper's LinearEnsemble probe)

In [ ]:
!python eval.py --probe probe.py --acts acts

## Step 2b: error analysis of the baseline (VAL only)
Which lies does the probe miss, which honest answers does it flag? Threshold = 5% false alarms on honest val answers.

In [ ]:
!python errors.py --probe probe.py --acts acts
from IPython.display import Markdown, display
display(Markdown(open('runs/errors_probe_val.md').read()))

## Step 3: the research loop (Claude Opus 5.5 proposes, eval decides)
Each iteration costs roughly $0.2–0.5 in API usage.

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('key loaded:', bool(os.environ['ANTHROPIC_API_KEY']))

In [ ]:
ITERS = 15
!python loop.py --acts acts --iters $ITERS

## Step 3b: comparison without an LLM
Plain random hyperparameter search (pool, C, layers, combination, family weights) with the same budget and the same keep rule. Tells us whether the LLM adds anything beyond tuning.

In [ ]:
!python search.py --acts acts --iters $ITERS

## Step 3c: robustness check (no GPU, no API)
Is the gain of the best probe (iter 9) over the baseline real? Paired bootstrap intervals on the original test families, plus 5 random family re-splits where every probe is refit. If Step 3b ran, its best probe is compared too. Needs `acts/` from Step 1 (if the Colab runtime was reset, rerun Step 1 first, ~13 min). About 1-2 min per re-split.

In [ ]:
!python robustness.py --acts acts --n_splits 5
from IPython.display import Markdown, display
display(Markdown(open('runs/robustness.md').read()))

## Step 4: results

In [ ]:
!python report.py
from IPython.display import Image, Markdown, display
display(Image('runs/progress.png'))
display(Markdown(open('runs/summary.md').read()))

In [ ]:
# error analysis of the best probe (val), then the final write-up on test
!python errors.py --probe probe.py --acts acts --out runs/errors_best_val.md
!python errors.py --probe probe.py --acts acts --split test --out runs/errors_best_test.md
display(Markdown(open('runs/errors_best_test.md').read()))

In [ ]:
# download everything (results + all tried probes)
!zip -qr research_loop_results.zip runs runs_search
from google.colab import files
files.download('research_loop_results.zip')